# Notebook 3 – Evaluate & Select Best Model

**Purpose:** Download all trained models from S3, evaluate them on the held-out test set, generate confusion matrices and a grouped bar chart, produce a ranked comparison table, identify the best-performing model, and upload all results back to S3.

**Prerequisites:** Run Notebooks 1 and 2 first.

## Step 1 – Load Config & Imports

In [ ]:
import json, os, boto3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

try:
    with open("config.json") as f:
        cfg = json.load(f)
    BUCKET       = cfg["bucket"]
    PREFIX       = cfg["prefix"]
    CLASS_NAMES  = cfg["class_names"]
    NUM_CLASSES  = len(CLASS_NAMES)
    MODELS_PREFIX = cfg["models_prefix"]
    RESULTS_PREFIX = cfg.get("results_prefix", f"{PREFIX}/results")
    print(f"✅ Config loaded — bucket: {BUCKET}, models_prefix: {MODELS_PREFIX}")
except FileNotFoundError:
    print("❌ config.json not found. Run Notebook 1 first.")
    raise

In [ ]:
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DOWNLOAD_DIR = "/tmp/asl_split"
MODELS_DIR   = "/tmp/asl_models_eval"
RESULTS_DIR  = "/tmp/asl_results"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
TEST_DIR    = os.path.join(DOWNLOAD_DIR, "test")

s3 = boto3.client("s3")
print("✅ Setup complete.")

## Step 2 – Download Models from S3

In [ ]:
# Download all .keras models
paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(Bucket=BUCKET, Prefix=MODELS_PREFIX)
model_keys = [obj["Key"] for page in pages for obj in page.get("Contents", [])
              if obj["Key"].endswith(".keras")]

print(f"Found {len(model_keys)} model files in S3:")
for k in model_keys:
    print(f"  {k}")

for key in model_keys:
    fname = os.path.basename(key)
    local_path = os.path.join(MODELS_DIR, fname)
    if not os.path.exists(local_path):
        s3.download_file(BUCKET, key, local_path)
        print(f"  ✅ Downloaded: {fname}")
    else:
        print(f"  ⏩ Already cached: {fname}")

model_files = [os.path.join(MODELS_DIR, os.path.basename(k)) for k in model_keys]
print(f"\n✅ {len(model_files)} model files ready.")

## Step 3 – Download Test Data (if not cached)

In [ ]:
from pathlib import Path

if os.path.exists(TEST_DIR) and len(list(Path(TEST_DIR).rglob("*.jpg"))) > 100:
    print(f"⏩ Test data already cached at {TEST_DIR}")
else:
    print("Downloading test data from S3 …")
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=BUCKET, Prefix=f"{PREFIX}/test")
    keys = [obj["Key"] for page in pages for obj in page.get("Contents", [])]
    print(f"  {len(keys)} test files")
    for i, key in enumerate(keys):
        rel = key[len(f"{PREFIX}/test"):].lstrip("/")
        dest = os.path.join(TEST_DIR, rel)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if not os.path.exists(dest):
            s3.download_file(BUCKET, key, dest)
        if (i + 1) % 2000 == 0:
            print(f"    … {i+1}/{len(keys)}")
    print("✅ Test data ready.")

## Step 4 – Evaluate Each Model

In [ ]:
import sys, os
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from mediapipe_utils import HandDetector


def evaluate_model(model_path, test_dir, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE):
    """Load a model and evaluate it on the test set."""
    model = tf.keras.models.load_model(model_path)
    model_name = os.path.basename(model_path).replace(".keras", "")

    # Landmark MLP uses feature vectors, not images
    if "landmark_mlp" in model_name:
        return evaluate_landmark_mlp(model, model_name, test_dir)

    gen = ImageDataGenerator(rescale=1./255)
    flow = gen.flow_from_directory(
        test_dir, target_size=image_size, batch_size=batch_size,
        class_mode="categorical", shuffle=False)

    probs = model.predict(flow, steps=len(flow), verbose=1)
    y_pred = np.argmax(probs, axis=1)
    y_true = flow.classes[:len(y_pred)]
    return y_true, y_pred, model_name


def evaluate_landmark_mlp(model, model_name, test_dir):
    """Evaluate landmark MLP by extracting features from test images."""
    import cv2
    label_map = {c: i for i, c in enumerate(CLASS_NAMES)}
    X, y_true = [], []

    with HandDetector() as det:
        if det.mode == 'unavailable':
            print("  ⚠️  MediaPipe unavailable — skipping Landmark MLP evaluation.")
            return np.array([]), np.array([]), model_name

        for cls in sorted(os.listdir(test_dir)):
            cls_dir = os.path.join(test_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue
                img = cv2.imread(os.path.join(cls_dir, fname))
                if img is None:
                    continue
                lms = det.process(img)
                if lms is None:
                    continue
                feats = np.array(lms, dtype=np.float32).flatten()
                X.append(feats)
                y_true.append(label_map.get(cls, 0))

    if not X:
        return np.array([]), np.array([]), model_name
    X_arr = np.array(X, dtype=np.float32)
    probs = model.predict(X_arr, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    return np.array(y_true), y_pred, model_name


# Ensure mediapipe_utils is importable from the notebook directory
sys.path.insert(0, os.path.dirname(os.path.abspath("mediapipe_utils.py")))
print("✅ Evaluation helpers ready.")

In [ ]:
comparison_records = []
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

for model_path in sorted(model_files):
    model_name = os.path.basename(model_path).replace(".keras", "")
    # Skip phase-specific checkpoints (e.g. *_p1_best / *_ft_best)
    if model_name.endswith("_best"):
        print(f"  ⏩ Skipping checkpoint: {model_name}")
        continue
    print(f"\n{'─'*60}")
    print(f"  Evaluating: {model_name}")
    try:
        y_true, y_pred, _ = evaluate_model(model_path, TEST_DIR)
        if len(y_true) == 0:
            print("  ⚠️  No valid samples — skipping")
            continue
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        continue

    acc = accuracy_score(y_true, y_pred)
    prec_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    prec_w     = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec_macro  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    rec_w      = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1_macro   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_w       = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"  Accuracy: {acc:.4f} | F1 macro: {f1_macro:.4f} | F1 weighted: {f1_w:.4f}")

    # Per-class report
    report = classification_report(y_true, y_pred,
                                   target_names=CLASS_NAMES[:max(y_true)+1] if len(y_true) else CLASS_NAMES,
                                   zero_division=0)
    report_path = os.path.join(RESULTS_DIR, f"report_{model_name}.txt")
    with open(report_path, "w") as f: f.write(report)

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    fig, ax = plt.subplots(figsize=(20, 18))
    labels = CLASS_NAMES[:cm.shape[0]]
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix (normalised) – {model_name}")
    plt.tight_layout()
    cm_path = os.path.join(PLOTS_DIR, f"cm_{model_name}.png")
    fig.savefig(cm_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  📊 Confusion matrix → {cm_path}")

    comparison_records.append({
        "model_name":         model_name,
        "accuracy":           round(acc, 4),
        "precision_macro":    round(prec_macro, 4),
        "precision_weighted": round(prec_w, 4),
        "recall_macro":       round(rec_macro, 4),
        "recall_weighted":    round(rec_w, 4),
        "f1_macro":           round(f1_macro, 4),
        "f1_weighted":        round(f1_w, 4),
    })

print(f"\n✅ Evaluated {len(comparison_records)} models.")

## Step 5 – Ranked Comparison Table

In [ ]:
df = pd.DataFrame(comparison_records).sort_values("accuracy", ascending=False).reset_index(drop=True)
print("\nModel Comparison (sorted by accuracy):")
print(df[["model_name","accuracy","f1_macro","f1_weighted"]].to_string(index=False))

# Save CSV and Markdown
csv_path = os.path.join(RESULTS_DIR, "model_comparison.csv")
md_path  = os.path.join(RESULTS_DIR, "model_comparison.md")
df.to_csv(csv_path, index=False)
try:
    df.to_markdown(md_path, index=False)
except ImportError:
    df.to_csv(md_path, sep="|", index=False)    # fallback
print(f"✅ Saved: {csv_path}")

## Step 6 – Grouped Bar Chart

In [ ]:
metrics_to_plot = ["accuracy", "f1_macro", "f1_weighted"]
x = range(len(df))
width = 0.8 / len(metrics_to_plot)

fig, ax = plt.subplots(figsize=(max(14, len(df)*1.2), 6))
for i, metric in enumerate(metrics_to_plot):
    offset = (i - len(metrics_to_plot)/2) * width + width/2
    ax.bar([xi + offset for xi in x], df[metric], width=width, label=metric)

ax.set_xticks(list(x))
ax.set_xticklabels(df["model_name"], rotation=45, ha="right", fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score"); ax.set_title("ASL Model Comparison")
ax.legend(); plt.tight_layout()
bar_path = os.path.join(PLOTS_DIR, "model_comparison_bar.png")
fig.savefig(bar_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"✅ Bar chart saved: {bar_path}")

## Step 7 – Identify Best Model

In [ ]:
best_row = df.iloc[0]
best_model_name = best_row["model_name"]

print("🏆 Best model:")
print(f"   Name       : {best_model_name}")
print(f"   Accuracy   : {best_row['accuracy']:.4f}")
print(f"   F1 macro   : {best_row['f1_macro']:.4f}")
print(f"   F1 weighted: {best_row['f1_weighted']:.4f}")

summary = {
    "best_model":   best_model_name,
    "metric":       "accuracy",
    "metrics":      best_row.to_dict(),
    "all_models":   df.to_dict(orient="records"),
}
summary_path = os.path.join(RESULTS_DIR, "results_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"✅ Summary saved: {summary_path}")

In [ ]:
# Update config.json with best model info
cfg["best_model_name"] = best_model_name
cfg["best_model_s3_uri"] = cfg.get("model_uris", {}).get(best_model_name, "")
with open("config.json", "w") as f:
    json.dump(cfg, f, indent=2)
print("✅ config.json updated with best model info.")

## Step 8 – Upload Results to S3

In [ ]:
import glob

# Upload all result files
result_files = (
    glob.glob(os.path.join(RESULTS_DIR, "*.csv"))  +
    glob.glob(os.path.join(RESULTS_DIR, "*.md"))   +
    glob.glob(os.path.join(RESULTS_DIR, "*.json")) +
    glob.glob(os.path.join(RESULTS_DIR, "*.txt"))  +
    glob.glob(os.path.join(PLOTS_DIR, "*.png"))
)

print(f"Uploading {len(result_files)} result files …")
for fpath in result_files:
    fname = os.path.basename(fpath)
    # plots go into plots/ sub-prefix
    subdir = "plots/" if fpath.endswith(".png") else ""
    s3_key = f"{RESULTS_PREFIX}/{subdir}{fname}"
    try:
        s3.upload_file(fpath, BUCKET, s3_key)
        print(f"  ✅ s3://{BUCKET}/{s3_key}")
    except Exception as e:
        print(f"  ❌ {fname}: {e}")

# Also upload updated config.json
s3.upload_file("config.json", BUCKET, f"{PREFIX}/config.json")
print("\n✅ All results uploaded.")

---
## ✅ Notebook 3 Complete

Your S3 bucket now contains a full evaluation report:

```
s3://<BUCKET>/<PREFIX>/results/
    model_comparison.csv          ← All models ranked by accuracy
    model_comparison.md           ← Markdown table
    results_summary.json          ← Best model + all metrics
    report_<model_name>.txt       ← Per-class classification report (one per model)
    plots/
        model_comparison_bar.png  ← Grouped bar chart
        cm_<model_name>.png       ← Normalised confusion matrix (one per model)
```

The best model has been identified and recorded in `config.json`.

### Next steps
- Use the best model path from `config.json["best_model_s3_uri"]` in your web app (`MODEL_PATH` env var)
- Set `MODEL_TYPE` to match the model name (e.g. `raw`, `cropped`, `skeleton`, or `landmark`)
- Run `docker compose up -d` or deploy to Cloud Run / Render using the Dockerfile from the `New` repository